In [6]:
import spacy

try:
    nlp = spacy.load("en_core_web_lg", disable=['parser'])
    nlp.max_length = 30000000 # Increase max length to allow processing large text files
except OSError:
    print("Downloading spaCy model 'en_core_web_lg'...")
    from spacy.cli import download
    download("en_core_web_lg")
    nlp = spacy.load("en_core_web_lg", disable=['parser'])
    nlp.max_length = 30000000


In [7]:
import json
import numpy as np
import os

def is_emoji(text):
    for char in text:
        # Check against common Unicode emoji ranges
        if (0x1F600 <= ord(char) <= 0x1F64F or
            0x1F300 <= ord(char) <= 0x1F5FF or
            0x1F680 <= ord(char) <= 0x1F6FF or
            0x2600 <= ord(char) <= 0x26FF or
            0x2700 <= ord(char) <= 0x27BF or
            0x1F900 <= ord(char) <= 0x1F9FF or
            0x1FA70 <= ord(char) <= 0x1FAFF):
            return True
    return False

def process_celeb_data(data):
    followers = data['text']
    celeb_id = data['id']
    vector_dict = {"PERSON":0, "NORP":0, "FAC":0, "ORG":0, "GPE":0, "LOC":0,"PRODUCT":0,
                   "EVENT":0,"WORK_OF_ART":0,"LAW":0,"LANGUAGE":0,"DATE":0,"TIME":0,
                   "PERCENT":0,"MONEY":0,"QUANTITY":0,"ORDINAL":0,"CARDINAL":0, 
                   "ADJ": 0, "ADP": 0, "ADV": 0, "AUX": 0, "CONJ": 0, "CCONJ": 0, 
                  "DET": 0, "INTJ": 0, "NOUN": 0, "NUM": 0, "PART": 0, "PRON": 0,
                  "PROPN": 0, "PUNCT": 0, "SCONJ": 0, "SYM": 0, "VERB": 0, "X": 0, 
                   "SPACE": 0, "STOP_WORDS": 0, "AVG_TWEET_LEN": 0, "MENTIONS": 0, "HASHTAGS": 0, 
                  "LINKS": 0, "EMOJI": 0}

    tweet_length = 0
    count_tweets = 0
    word_vecs = []

    all_followers_text = []
    for follower in followers:
        # follower is a list of tweets
        for tweet in follower:
            tweet_length += len(tweet)
            count_tweets += 1
        all_followers_text.append(" ".join(follower))

    full_text = " ".join(all_followers_text)
    doc = nlp(full_text)
    num_tokens = len(doc)

    for token in doc:
        if token.text.startswith('#') and len(token.text) > 1:
            vector_dict['HASHTAGS'] += 1
        elif token.text.startswith('@') and len(token.text) > 1:
            vector_dict['MENTIONS'] += 1
        elif token.like_url:
            vector_dict['LINKS'] += 1
        elif is_emoji(token.text):
            vector_dict['EMOJI'] += 1

        if token.pos_ in vector_dict:
            vector_dict[token.pos_] += 1

        if token.is_stop:
            vector_dict['STOP_WORDS'] += 1
        if token.has_vector:
            word_vecs.append(token.vector)

    for entity in doc.ents:
        if entity.label_ in vector_dict:
            vector_dict[entity.label_] += 1

    if num_tokens > 0:
        vector_dict = dict(map(lambda feature: (feature[0], feature[1] / num_tokens), vector_dict.items()))

    if count_tweets > 0:
        vector_dict['AVG_TWEET_LEN'] = tweet_length / count_tweets
    else:
        vector_dict['AVG_TWEET_LEN'] = 0

    word_vec_array = np.array(word_vecs)
    if word_vec_array.size > 0:
        wv = np.mean(word_vec_array, axis=0)
    else:
        wv = np.zeros(300) # en_core_web_lg vectors are 300d

    word_vec_list = wv.tolist()
    total_list = [celeb_id] + word_vec_list + list(vector_dict.values())
    return total_list




In [8]:
job_dict = {'sports': 0, 'creator': 1, 'politics': 2, 'performer': 3}
gender_dict = {'male': 0, 'female': 1}
label_dict = {}
with open('./traindata/labels.ndjson', encoding='utf-8') as file:
    labels = file.readlines()
    for line in labels:
        celeb = json.loads(line)
        label_dict[str(celeb['id'])] = [celeb['birthyear'], gender_dict[celeb['gender']], job_dict[celeb['occupation']]]


In [9]:
def process_ndjson(input_file, output_dir):
    feature_vecs = []
    label_vecs = []

    print(f"Reading data from {input_file}...")
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            data = json.loads(line)
            celeb_id = str(data['id'])

            if celeb_id in label_dict:
                label_vecs.append(label_dict[celeb_id])
                try:
                    print(f"Processing celeb ID: {celeb_id}")
                    vector_list = process_celeb_data(data)
                    feature_vecs.append(vector_list)
                except Exception as e:
                    print(f"Error processing {celeb_id}: {e}")
            else:
                print(f"No label found for {celeb_id}")

    if feature_vecs:
        feature_vec_array = np.array(feature_vecs)
        label_vec_array = np.array(label_vecs)

        if not os.path.exists(output_dir):
            os.makedirs(output_dir)

        np.save(os.path.join(output_dir, 'features.npy'), feature_vec_array)
        np.save(os.path.join(output_dir, 'labels.npy'), label_vec_array)
        print(f"Successfully saved features and labels to {output_dir}")
    else:
        print(f"No features processed")
    


In [10]:
# Read directly from the traindata folder and output to Testing/celeb_files
process_ndjson('./traindata/follower-feeds.ndjson', './Testing/celeb_files/')


Reading data from ./traindata/follower-feeds.ndjson...
Processing celeb ID: 1
Processing celeb ID: 2
Processing celeb ID: 3
Processing celeb ID: 4
Processing celeb ID: 5
Processing celeb ID: 6
Processing celeb ID: 7
Processing celeb ID: 8
Processing celeb ID: 9
Processing celeb ID: 10
Processing celeb ID: 11
Processing celeb ID: 12
Processing celeb ID: 13
Processing celeb ID: 14
Processing celeb ID: 15
Processing celeb ID: 16
Processing celeb ID: 17


KeyboardInterrupt: 